# 01_da_agreement_template

Standalone **Data Agreement Intake / Usage Boundary** notebook. It writes one append-only row per agreement version to `METADATA_DATA_AGREEMENT`. Governance classification and review remain in `04_gov_*`.


In [ ]:
%run 00_env_config


In [ ]:
from IPython.display import clear_output
from fabricops_kit import (
    collect_agreement_metadata,
    commit_agreement_metadata,
    create_agreement_form,
    load_agreements,
    read_agreement_form,
    setup_data_agreement_tables,
)


## Defensive metadata readiness check

`00_env_config` owns environment bootstrap and creates or checks the agreement and steward tables in the configured metadata lakehouse. This defensive call is intentionally idempotent. The intake form still requires real active `METADATA_DATA_STEWARD` rows; no fake people are seeded.


In [ ]:
setup_data_agreement_tables(spark=spark, config=CONFIG, env=ENV)


In [ ]:
agreement_form = create_agreement_form(spark=spark, config=CONFIG, env=ENV)


In [ ]:
def on_commit_clicked(_):
    with agreement_form["output"]:
        clear_output()
        try:
            latest = load_agreements(CONFIG, ENV, spark_session=spark, missing_ok=True)
            intake_mode = "update" if agreement_form["mode"].value == "Update Existing Agreement" else "create"
            selected = agreement_form["existing_agreement"].value if intake_mode == "update" else None
            if intake_mode == "update" and not selected:
                raise ValueError("Update mode selected, but no existing agreement was chosen.")
            metadata = collect_agreement_metadata(widget_values=read_agreement_form(agreement_form), mode=intake_mode, existing_rows=latest, selected_agreement=selected, config=CONFIG, env=ENV)
            summary = commit_agreement_metadata(spark=spark, config=CONFIG, env=ENV, agreement_metadata=metadata)
            print("Data agreement committed successfully.")
            print(f"- Agreement ID: {summary['agreement_id']}")
            print(f"- Contract Version: {summary['contract_version']}")
            print(f"- Status: {summary['agreement_status']}")
            print(f"- Review Status: {summary['review_status']}")
            print(f"- Expiry Date: {summary['expiry_date']}")
            print(f"- Committed By: {summary['committed_by']}")
            print(f"- Committed At: {summary['committed_at']}")
            print(f"- Table Updated: {summary['table_updated']}")
        except Exception as exc:
            print(f"Commit failed: {exc}")

agreement_form["commit_button"].on_click(on_commit_clicked)
